# 32 Entity Deduplication

Clusters Phase 31 alignment units into canonical person entities.

**Input:** 

**Output:** 
-  — one row per canonical entity
-  — membership mapping
-  — run statistics

**Strategies:**
1.  (high confidence): rows sharing a non-empty 
2.  (medium confidence): same  key
3.  (low confidence): no cluster partner found

In [ ]:
from pathlib import Path
from datetime import UTC, datetime
import os
import sys

import pandas as pd

repo_root = Path.cwd()
if not (repo_root / "speakermining").exists():
    # Notebook may be opened from the notebooks folder.
    for parent in [repo_root] + list(repo_root.parents):
        if (parent / "speakermining").exists():
            repo_root = parent
            break

os.chdir(repo_root)
src_path = repo_root / "speakermining" / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

In [ ]:
from process.entity_deduplication.orchestrator import run_phase32

summary = run_phase32()
print("Phase 32 complete.")

In [ ]:
import json
print(json.dumps(summary, indent=2, ensure_ascii=False))

## Cluster distribution

In [ ]:
import pandas as pd

dedup_persons = pd.read_csv("data/32_entity_deduplication/dedup_persons.csv", dtype=str).fillna("")

print("Canonical entities by strategy:")
print(dedup_persons["cluster_strategy"].value_counts().to_string())
print()
print("Canonical entities by confidence:")
print(dedup_persons["cluster_confidence"].value_counts().to_string())

In [ ]:
# Largest clusters (most alignment units per canonical entity)
top_clusters = (
    dedup_persons
    .assign(cluster_size=lambda df: df["cluster_size"].astype(int))
    .nlargest(20, "cluster_size")
    [["canonical_entity_id", "cluster_size", "cluster_strategy", "wikidata_id", "canonical_label"]]
)
print(top_clusters.to_string(index=False))

## Wikidata-matched clusters

Clusters with  strategy have the highest confidence.

In [ ]:
wd_clusters = dedup_persons[dedup_persons["cluster_strategy"] == "wikidata_qid_match"].copy()
wd_clusters["cluster_size"] = wd_clusters["cluster_size"].astype(int)
print(f"Wikidata-matched clusters: {len(wd_clusters)}")
print(f"Total alignment units covered: {wd_clusters['cluster_size'].sum()}")
print()
print(wd_clusters.sort_values("cluster_size", ascending=False).head(10)[
    ["canonical_label", "wikidata_id", "cluster_size"]
].to_string(index=False))

## Next step: Step 322 Manual Review

Medium-confidence clusters () are candidates for manual inspection.
Load , filter ,
and review in OpenRefine or a spreadsheet tool before proceeding to Phase 4.